# MGCI Inference and Calculation Pipeline

**SDG Indicator 15.4.2: Mountain Green Cover Index**

This notebook runs inference on all patches and calculates MGCI using multiple methods.

### Pipeline Overview
1. Load trained model from Notebook 2
2. Run inference on all patches
3. Calculate MGCI using NDVI threshold method
4. Calculate MGCI using model predictions
5. Apply true surface area correction (slope-based)
6. Stratify results by Kapos elevation class
7. Generate publication-ready figures

**Key Output:** MGCI comparison table and figures for SDG reporting

## 1. Setup and Configuration

In [ ]:
!pip install -q rasterio segmentation-models-pytorch
print("Dependencies installed")

In [ ]:
import os
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm
from datetime import datetime
import rasterio
import torch
import segmentation_models_pytorch as smp
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Libraries imported | Device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

In [ ]:
class Config:
    """Configuration for MGCI inference."""
    
    # Paths - UPDATE FOR YOUR SETUP
    DATA_DIR = '/content/drive/MyDrive/MGCI_Jizan_2024'
    MODEL_PATH = '/content/drive/MyDrive/MGCI_Jizan_2024/model_best.pth'
    OUTPUT_DIR = '/content/outputs'

    # Region info
    REGION_NAME = 'Jizan'
    YEAR = 2024

    # Parameters
    MOUNTAIN_THRESHOLD = 300  # meters (Kapos Class 6 minimum)
    NDVI_THRESHOLD = 0.30
    RESOLUTION = 10  # meters

    # Band indices (7-band GeoTIFF stack from Notebook 1)
    BLUE_BAND = 0
    GREEN_BAND = 1
    RED_BAND = 2
    NIR_BAND = 3
    ELEV_BAND = 4
    SLOPE_BAND = 5
    LABEL_BAND = 6

    # Model input bands
    INPUT_BANDS = [0, 1, 2, 3, 4, 5]


os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Data directory: {Config.DATA_DIR}")
print(f"Model path: {Config.MODEL_PATH}")

## 2. Load Model and Data

In [ ]:
# Load trained model
print("Loading trained model...")

checkpoint = torch.load(Config.MODEL_PATH, map_location=device, weights_only=False)

MEANS = checkpoint['means']
STDS = checkpoint['stds']

model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights=None,
    in_channels=6,
    classes=1,
    activation=None
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded (epoch {checkpoint['epoch']+1}, val_iou={checkpoint['val_iou']:.4f})")

In [ ]:
# Load all patches
all_files = sorted(glob.glob(f'{Config.DATA_DIR}/*.tif'))
print(f"Found {len(all_files)} patches")

if len(all_files) == 0:
    raise FileNotFoundError(f"No .tif files found in {Config.DATA_DIR}")

## 3. Process All Patches

In [ ]:
print("Processing all patches...")

# Accumulators for planimetric area (pixel counts)
total_pixels = 0
mountain_pixels = 0
ndvi_green_mtn_pixels = 0
model_green_mtn_pixels = 0

# Accumulators for true surface area (slope-corrected)
total_surface_area = 0
mountain_surface_area = 0
ndvi_green_mtn_surface = 0
model_green_mtn_surface = 0

# Per-patch results for visualization
patch_results = []

# Pixel area in m²
pixel_area = Config.RESOLUTION ** 2

for f in tqdm(all_files, desc="Processing"):
    try:
        with rasterio.open(f) as src:
            data = src.read().astype(np.float32)

        # Extract bands
        elevation = data[Config.ELEV_BAND]
        slope_deg = data[Config.SLOPE_BAND]
        nir = data[Config.NIR_BAND]
        red = data[Config.RED_BAND]
        veg_label = data[Config.LABEL_BAND]

        # Calculate NDVI
        ndvi = (nir - red) / (nir + red + 1e-6)
        ndvi_veg = (ndvi > Config.NDVI_THRESHOLD).astype(float)

        # Model prediction
        inputs = data[Config.INPUT_BANDS]
        inputs_norm = (inputs - np.array(MEANS).reshape(-1,1,1)) / np.array(STDS).reshape(-1,1,1)
        inputs_tensor = torch.tensor(inputs_norm).unsqueeze(0).float().to(device)

        with torch.no_grad():
            pred = torch.sigmoid(model(inputs_tensor)).cpu().numpy().squeeze()
        model_veg = (pred > 0.5).astype(float)

        # Mountain mask
        mountain = elevation >= Config.MOUNTAIN_THRESHOLD

        # True surface area calculation: Surface area = Planimetric area / cos(slope)
        slope_rad = np.radians(slope_deg)
        cos_slope = np.cos(slope_rad)
        cos_slope = np.clip(cos_slope, 0.1, 1.0)  # Avoid division issues
        surface_area_factor = 1 / cos_slope

        # Patch statistics
        n_pixels = elevation.size
        n_mountain = mountain.sum()

        # Planimetric (pixel counts)
        total_pixels += n_pixels
        mountain_pixels += n_mountain
        ndvi_green_mtn_pixels += (mountain & (ndvi_veg == 1)).sum()
        model_green_mtn_pixels += (mountain & (model_veg == 1)).sum()

        # True surface area (slope-corrected)
        patch_surface = (surface_area_factor * pixel_area).sum()
        patch_mtn_surface = (surface_area_factor[mountain] * pixel_area).sum()

        ndvi_green_in_mtn = mountain & (ndvi_veg == 1)
        model_green_in_mtn = mountain & (model_veg == 1)

        total_surface_area += patch_surface
        mountain_surface_area += patch_mtn_surface
        ndvi_green_mtn_surface += (surface_area_factor[ndvi_green_in_mtn] * pixel_area).sum() if ndvi_green_in_mtn.any() else 0
        model_green_mtn_surface += (surface_area_factor[model_green_in_mtn] * pixel_area).sum() if model_green_in_mtn.any() else 0

        # Store per-patch results
        if n_mountain > 0:
            patch_results.append({
                'file': f,
                'mountain_pct': n_mountain / n_pixels * 100,
                'ndvi_mgci': (mountain & (ndvi_veg == 1)).sum() / n_mountain * 100,
                'model_mgci': (mountain & (model_veg == 1)).sum() / n_mountain * 100,
                'slope_mean': slope_deg[mountain].mean() if mountain.any() else 0,
                'elev_mean': elevation[mountain].mean() if mountain.any() else 0
            })
    except Exception as e:
        print(f"Error processing {f}: {e}")
        continue

print(f"Processed {len(all_files)} patches")

## 4. Calculate MGCI Results

In [ ]:
# Convert to km²
total_area_km2 = total_pixels * pixel_area / 1e6
mountain_area_km2 = mountain_pixels * pixel_area / 1e6
ndvi_green_km2 = ndvi_green_mtn_pixels * pixel_area / 1e6
model_green_km2 = model_green_mtn_pixels * pixel_area / 1e6

# True surface area (km²)
total_surface_km2 = total_surface_area / 1e6
mountain_surface_km2 = mountain_surface_area / 1e6
ndvi_green_surface_km2 = ndvi_green_mtn_surface / 1e6
model_green_surface_km2 = model_green_mtn_surface / 1e6

# MGCI calculations
mgci_ndvi_planimetric = (ndvi_green_km2 / mountain_area_km2 * 100) if mountain_area_km2 > 0 else 0
mgci_model_planimetric = (model_green_km2 / mountain_area_km2 * 100) if mountain_area_km2 > 0 else 0
mgci_ndvi_surface = (ndvi_green_surface_km2 / mountain_surface_km2 * 100) if mountain_surface_km2 > 0 else 0
mgci_model_surface = (model_green_surface_km2 / mountain_surface_km2 * 100) if mountain_surface_km2 > 0 else 0

# Surface area correction factor
surface_correction = (mountain_surface_km2 / mountain_area_km2 - 1) * 100 if mountain_area_km2 > 0 else 0

print("=" * 70)
print("MGCI CALCULATION RESULTS")
print("=" * 70)

print(f"\nArea Statistics:")
print(f"  Total Area (Planimetric):      {total_area_km2:>10,.2f} km²")
print(f"  Mountain Area (Planimetric):   {mountain_area_km2:>10,.2f} km²")
print(f"  Mountain Area (True Surface):  {mountain_surface_km2:>10,.2f} km²")
print(f"  Surface Area Correction:       {surface_correction:>10.2f}%")

print(f"\nGreen Mountain Area:")
print(f"  NDVI-based (Planimetric):      {ndvi_green_km2:>10,.2f} km²")
print(f"  NDVI-based (True Surface):     {ndvi_green_surface_km2:>10,.2f} km²")
print(f"  Model-based (Planimetric):     {model_green_km2:>10,.2f} km²")
print(f"  Model-based (True Surface):    {model_green_surface_km2:>10,.2f} km²")

print(f"\n{'Method':<40} {'MGCI (%)':>12}")
print("-" * 55)
print(f"{'1. NDVI (Planimetric Area)':<40} {mgci_ndvi_planimetric:>12.2f}")
print(f"{'2. NDVI (True Surface Area)':<40} {mgci_ndvi_surface:>12.2f}")
print(f"{'3. Model (Planimetric Area)':<40} {mgci_model_planimetric:>12.2f}")
print(f"{'4. Model (True Surface Area)':<40} {mgci_model_surface:>12.2f}")
print("-" * 55)
print(f"\nDifference (Model - NDVI): {mgci_model_planimetric - mgci_ndvi_planimetric:+.2f}%")
print(f"Surface Correction Impact: {mgci_ndvi_surface - mgci_ndvi_planimetric:+.2f}%")
print("=" * 70)

## 5. MGCI by Kapos Elevation Class

In [ ]:
print("Calculating MGCI by Kapos elevation class...")

# Kapos class definitions (elevation ranges in meters)
kapos_classes = {
    6: {'name': 'Class 6: Foothills', 'min': 300, 'max': 1000},
    5: {'name': 'Class 5: Lower Montane', 'min': 1000, 'max': 1500},
    4: {'name': 'Class 4: Montane', 'min': 1500, 'max': 2500},
    3: {'name': 'Class 3: Upper Montane', 'min': 2500, 'max': 3500},
    2: {'name': 'Class 2: Alpine', 'min': 3500, 'max': 4500},
    1: {'name': 'Class 1: Nival', 'min': 4500, 'max': 9999},
}

# Initialize accumulators for each Kapos class
kapos_stats = {k: {
    'mountain_pixels': 0,
    'green_pixels_ndvi': 0,
    'green_pixels_model': 0,
    'surface_area': 0,
    'green_surface_ndvi': 0,
    'green_surface_model': 0
} for k in kapos_classes.keys()}

for f in tqdm(all_files, desc="Kapos Analysis"):
    try:
        with rasterio.open(f) as src:
            data = src.read().astype(np.float32)

        inputs = data[Config.INPUT_BANDS]
        elevation = data[Config.ELEV_BAND]
        slope_deg = data[Config.SLOPE_BAND]
        nir = data[Config.NIR_BAND]
        red = data[Config.RED_BAND]

        mountain = elevation >= Config.MOUNTAIN_THRESHOLD

        # NDVI calculation
        ndvi = (nir - red) / (nir + red + 1e-10)
        ndvi_veg = (ndvi > Config.NDVI_THRESHOLD).astype(np.uint8)

        # Model prediction
        x = torch.from_numpy(inputs).unsqueeze(0)
        for i in range(6):
            x[:, i] = (x[:, i] - MEANS[i]) / (STDS[i] + 1e-8)

        with torch.no_grad():
            pred = model(x.to(device))
            pred = torch.sigmoid(pred).cpu().numpy()[0, 0]
        model_veg = (pred > 0.5).astype(np.uint8)

        # Surface area factor
        slope_rad = np.deg2rad(np.clip(slope_deg, 0, 89))
        surface_factor = 1.0 / np.cos(slope_rad)
        pixel_area_m2 = Config.RESOLUTION ** 2

        # Accumulate stats for each Kapos class
        for class_id, class_info in kapos_classes.items():
            class_mask = mountain & (elevation >= class_info['min']) & (elevation < class_info['max'])

            if class_mask.any():
                n_class_pixels = class_mask.sum()
                kapos_stats[class_id]['mountain_pixels'] += n_class_pixels
                kapos_stats[class_id]['green_pixels_ndvi'] += (class_mask & (ndvi_veg == 1)).sum()
                kapos_stats[class_id]['green_pixels_model'] += (class_mask & (model_veg == 1)).sum()

                class_surface = (surface_factor[class_mask] * pixel_area_m2).sum()
                kapos_stats[class_id]['surface_area'] += class_surface

                green_ndvi_mask = class_mask & (ndvi_veg == 1)
                green_model_mask = class_mask & (model_veg == 1)
                kapos_stats[class_id]['green_surface_ndvi'] += (surface_factor[green_ndvi_mask] * pixel_area_m2).sum() if green_ndvi_mask.any() else 0
                kapos_stats[class_id]['green_surface_model'] += (surface_factor[green_model_mask] * pixel_area_m2).sum() if green_model_mask.any() else 0

    except Exception as e:
        continue

# Print results table
print("\n" + "=" * 90)
print("MGCI BY KAPOS ELEVATION CLASS (True Surface Area)")
print("=" * 90)

print(f"\n{'Kapos Class':<25} {'Elevation':<15} {'Area (km²)':<12} {'MGCI-NDVI':<12} {'MGCI-Model':<12}")
print("-" * 90)

total_area_kapos = 0
total_green_ndvi = 0
total_green_model = 0
results_by_class = []

for class_id in sorted(kapos_classes.keys(), reverse=True):
    stats = kapos_stats[class_id]
    class_info = kapos_classes[class_id]

    if stats['surface_area'] > 0:
        area_km2 = stats['surface_area'] / 1e6
        mgci_ndvi = (stats['green_surface_ndvi'] / stats['surface_area']) * 100
        mgci_model = (stats['green_surface_model'] / stats['surface_area']) * 100

        total_area_kapos += stats['surface_area']
        total_green_ndvi += stats['green_surface_ndvi']
        total_green_model += stats['green_surface_model']

        elev_range = f"{class_info['min']}-{class_info['max']}m"
        print(f"{class_info['name']:<25} {elev_range:<15} {area_km2:>10.2f}   {mgci_ndvi:>10.2f}%  {mgci_model:>10.2f}%")

        results_by_class.append({
            'class': class_id,
            'name': class_info['name'],
            'elevation_range': elev_range,
            'area_km2': area_km2,
            'mgci_ndvi': mgci_ndvi,
            'mgci_model': mgci_model
        })

print("-" * 90)

if total_area_kapos > 0:
    total_area_km2_kapos = total_area_kapos / 1e6
    total_mgci_ndvi = (total_green_ndvi / total_area_kapos) * 100
    total_mgci_model = (total_green_model / total_area_kapos) * 100
    print(f"{'TOTAL (All Mountain)':<25} {'>300m':<15} {total_area_km2_kapos:>10.2f}   {total_mgci_ndvi:>10.2f}%  {total_mgci_model:>10.2f}%")

print("\nMGCI by Kapos class calculated")

## 6. Visualizations

In [ ]:
# MGCI Comparison Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['NDVI\n(Planimetric)', 'NDVI\n(True Surface)', 'Model\n(Planimetric)', 'Model\n(True Surface)']
values = [mgci_ndvi_planimetric, mgci_ndvi_surface, mgci_model_planimetric, mgci_model_surface]
colors = ['#90EE90', '#228B22', '#ADD8E6', '#4169E1']

bars = ax.bar(methods, values, color=colors, edgecolor='black', linewidth=1.5)

ax.set_ylabel('MGCI (%)', fontsize=12)
ax.set_title(f'MGCI Comparison - {Config.REGION_NAME} ({Config.YEAR})', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(values) * 1.2)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/mgci_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/mgci_comparison.png")

In [ ]:
# Per-patch MGCI distribution analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ndvi_mgcis = [p['ndvi_mgci'] for p in patch_results]
model_mgcis = [p['model_mgci'] for p in patch_results]
slopes = [p['slope_mean'] for p in patch_results]
elevs = [p['elev_mean'] for p in patch_results]

# 1. NDVI vs Model MGCI scatter
axes[0, 0].scatter(ndvi_mgcis, model_mgcis, alpha=0.5, s=20, c='steelblue')
axes[0, 0].plot([0, 100], [0, 100], 'r--', lw=2, label='1:1 line')
axes[0, 0].set_xlabel('NDVI MGCI (%)', fontsize=11)
axes[0, 0].set_ylabel('Model MGCI (%)', fontsize=11)
axes[0, 0].set_title('NDVI vs Model MGCI (per patch)', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_xlim(0, 100)
axes[0, 0].set_ylim(0, 100)

# 2. MGCI distribution histogram
axes[0, 1].hist(ndvi_mgcis, bins=30, alpha=0.6, label='NDVI', color='#3498db', edgecolor='black')
axes[0, 1].hist(model_mgcis, bins=30, alpha=0.6, label='Model', color='#e74c3c', edgecolor='black')
axes[0, 1].axvline(np.mean(ndvi_mgcis), color='#3498db', linestyle='--', lw=2)
axes[0, 1].axvline(np.mean(model_mgcis), color='#e74c3c', linestyle='--', lw=2)
axes[0, 1].set_xlabel('MGCI (%)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('MGCI Distribution', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. MGCI vs Slope
axes[1, 0].scatter(slopes, ndvi_mgcis, alpha=0.5, s=20, label='NDVI', c='#3498db')
axes[1, 0].scatter(slopes, model_mgcis, alpha=0.5, s=20, label='Model', c='#e74c3c')
axes[1, 0].set_xlabel('Mean Slope (degrees)', fontsize=11)
axes[1, 0].set_ylabel('MGCI (%)', fontsize=11)
axes[1, 0].set_title('MGCI vs Slope', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. MGCI vs Elevation
axes[1, 1].scatter(elevs, ndvi_mgcis, alpha=0.5, s=20, label='NDVI', c='#3498db')
axes[1, 1].scatter(elevs, model_mgcis, alpha=0.5, s=20, label='Model', c='#e74c3c')
axes[1, 1].set_xlabel('Mean Elevation (m)', fontsize=11)
axes[1, 1].set_ylabel('MGCI (%)', fontsize=11)
axes[1, 1].set_title('MGCI vs Elevation', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle(f'MGCI Analysis - {Config.REGION_NAME} ({len(patch_results)} patches)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/mgci_analysis.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/mgci_analysis.png")

In [ ]:
# MGCI by Kapos class visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

valid_classes = [r for r in results_by_class if r['area_km2'] > 0]
class_names = [r['name'].replace('Class ', 'C') for r in valid_classes]
mgci_ndvi_vals = [r['mgci_ndvi'] for r in valid_classes]
mgci_model_vals = [r['mgci_model'] for r in valid_classes]
areas = [r['area_km2'] for r in valid_classes]

x = np.arange(len(class_names))
width = 0.35

# Bar chart - MGCI by class
ax1 = axes[0]
bars1 = ax1.bar(x - width/2, mgci_ndvi_vals, width, label='NDVI', color='lightgreen', edgecolor='green')
bars2 = ax1.bar(x + width/2, mgci_model_vals, width, label='U-Net Model', color='forestgreen', edgecolor='darkgreen')
ax1.set_xlabel('Kapos Elevation Class', fontsize=11)
ax1.set_ylabel('MGCI (%)', fontsize=11)
ax1.set_title('MGCI by Kapos Elevation Class', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(class_names, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

for bar in bars1:
    height = bar.get_height()
    ax1.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax1.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

# Pie chart - Area distribution
ax2 = axes[1]
colors = plt.cm.YlGn(np.linspace(0.3, 0.9, len(valid_classes)))
wedges, texts, autotexts = ax2.pie(areas, labels=class_names, autopct='%1.1f%%',
                                    colors=colors, startangle=90)
ax2.set_title('Mountain Area Distribution by Kapos Class', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/mgci_by_kapos_class.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/mgci_by_kapos_class.png")

In [ ]:
# Sample prediction comparison
print("Generating sample prediction comparisons...")

def normalize_band(band, pmin=2, pmax=98):
    """Normalize band for visualization."""
    valid = band[band > 0]
    if len(valid) == 0:
        return np.zeros_like(band)
    vmin, vmax = np.percentile(valid, [pmin, pmax])
    return np.clip((band - vmin) / (vmax - vmin + 1e-8), 0, 1)

sorted_patches = sorted(patch_results, key=lambda x: x['model_mgci'])
n = len(sorted_patches)
selected_indices = [0, n//4, n//2, 3*n//4, n-1, n//3]
selected_patches = [sorted_patches[i] for i in selected_indices if i < n]

fig, axes = plt.subplots(len(selected_patches), 5, figsize=(24, 4*len(selected_patches)))

for idx, patch_info in enumerate(selected_patches):
    with rasterio.open(patch_info['file']) as src:
        data = src.read().astype(np.float32)

    elevation = data[Config.ELEV_BAND]
    slope_deg = data[Config.SLOPE_BAND]
    nir = data[Config.NIR_BAND]
    red = data[Config.RED_BAND]
    inputs = data[Config.INPUT_BANDS]

    mountain = elevation >= Config.MOUNTAIN_THRESHOLD
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndvi_veg = ndvi > Config.NDVI_THRESHOLD

    inputs_norm = (inputs - np.array(MEANS).reshape(-1,1,1)) / np.array(STDS).reshape(-1,1,1)
    inputs_tensor = torch.tensor(inputs_norm).unsqueeze(0).float().to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(inputs_tensor)).cpu().numpy().squeeze()
    model_veg = pred > 0.5

    # RGB composite
    rgb = np.stack([
        normalize_band(data[Config.RED_BAND]),
        normalize_band(data[Config.GREEN_BAND]),
        normalize_band(data[Config.BLUE_BAND])
    ], axis=-1)

    axes[idx, 0].imshow(rgb)
    axes[idx, 0].set_title('RGB' if idx == 0 else '', fontsize=12)
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(elevation, cmap='terrain')
    axes[idx, 1].set_title('Elevation' if idx == 0 else '', fontsize=12)
    axes[idx, 1].axis('off')

    axes[idx, 2].imshow(ndvi_veg, cmap='Greens')
    axes[idx, 2].set_title('NDVI Vegetation' if idx == 0 else '', fontsize=12)
    axes[idx, 2].axis('off')

    axes[idx, 3].imshow(model_veg, cmap='Greens')
    axes[idx, 3].set_title('Model Prediction' if idx == 0 else '', fontsize=12)
    axes[idx, 3].axis('off')

    # MGCI overlay
    h, w = mountain.shape
    overlay = np.zeros((h, w, 3))
    overlay[~mountain] = [0.9, 0.9, 0.9]
    overlay[mountain & ~model_veg] = [0.8, 0.7, 0.5]
    overlay[mountain & model_veg] = [0.1, 0.6, 0.1]
    axes[idx, 4].imshow(overlay)
    axes[idx, 4].set_title(f'MGCI: {patch_info["model_mgci"]:.1f}%' if idx == 0 else f'{patch_info["model_mgci"]:.1f}%', fontsize=12)
    axes[idx, 4].axis('off')

plt.suptitle('Sample Patches - NDVI vs Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/sample_patches_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/sample_patches_comparison.png")

## 7. Save Results

In [ ]:
# Helper function for JSON serialization
def convert_numpy_types(obj):
    """Convert numpy types to native Python types for JSON serialization."""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(elem) for elem in obj]
    else:
        return obj


# Save MGCI results
results = {
    'region': Config.REGION_NAME,
    'year': Config.YEAR,
    'n_patches': len(all_files),
    'total_area_km2': total_area_km2,
    'mountain_area_planimetric_km2': mountain_area_km2,
    'mountain_area_surface_km2': mountain_surface_km2,
    'surface_correction_pct': surface_correction,
    'mgci': {
        'ndvi_planimetric': mgci_ndvi_planimetric,
        'ndvi_surface': mgci_ndvi_surface,
        'model_planimetric': mgci_model_planimetric,
        'model_surface': mgci_model_surface
    },
    'green_mountain_area_km2': {
        'ndvi_planimetric': ndvi_green_km2,
        'ndvi_surface': ndvi_green_surface_km2,
        'model_planimetric': model_green_km2,
        'model_surface': model_green_surface_km2
    }
}

results_serializable = convert_numpy_types(results)

with open(f'{Config.OUTPUT_DIR}/mgci_results.json', 'w') as f:
    json.dump(results_serializable, f, indent=2)

# Save Kapos class results
kapos_results = {
    'by_class': results_by_class,
    'total': {
        'area_km2': total_area_km2_kapos if 'total_area_km2_kapos' in dir() else 0,
        'mgci_ndvi': total_mgci_ndvi if 'total_mgci_ndvi' in dir() else 0,
        'mgci_model': total_mgci_model if 'total_mgci_model' in dir() else 0
    }
}

kapos_results_serializable = convert_numpy_types(kapos_results)

with open(f'{Config.OUTPUT_DIR}/mgci_by_kapos_class.json', 'w') as f:
    json.dump(kapos_results_serializable, f, indent=2)

print(f"Results saved:")
print(f"  {Config.OUTPUT_DIR}/mgci_results.json")
print(f"  {Config.OUTPUT_DIR}/mgci_by_kapos_class.json")

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("INFERENCE COMPLETE")
print("=" * 70)

print(f"\nRegion: {Config.REGION_NAME}")
print(f"Year: {Config.YEAR}")
print(f"Patches processed: {len(all_files)}")

print(f"\nMGCI Results (True Surface Area):")
print(f"  NDVI method:  {mgci_ndvi_surface:.2f}%")
print(f"  Model method: {mgci_model_surface:.2f}%")
print(f"  Difference:   {mgci_model_surface - mgci_ndvi_surface:+.2f}%")

print(f"\nOutput Files:")
print(f"  mgci_results.json")
print(f"  mgci_by_kapos_class.json")
print(f"  mgci_comparison.png")
print(f"  mgci_analysis.png")
print(f"  mgci_by_kapos_class.png")
print(f"  sample_patches_comparison.png")

print("=" * 70)